# Video search on a laptop with EmbeddingGemma 2

This notebook walks the whole result: a text query finds a video clip, offline, on an Intel laptop's integrated
GPU; then the same vectors are asked which way time runs, and cannot tell.

**Before running it**, run the scripts in the README up to `quantize.py` (download, prepare, embed, evaluate,
export, quantize). The notebook reads `data/`, `models/` and `index/`, and the tables the scripts wrote to
`output/`.

In [ ]:
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, Markdown, display

from ov_model import OVEmbedder, igpu
from search import truncate
from smoke_export import sample_frames

CLIPS, INDEX, OUT = Path("data/clips"), Path("index"), Path("output")
spec = json.loads(Path("clips.json").read_text())
clips, queries = spec["clips"], spec["queries"]


def section(path, start, stop=None):
    """Show the part of a saved Markdown table from the line starting with `start`."""
    text = Path(path).read_text()
    part = text[text.index(start):]
    if stop:
        part = part[: part.index(stop)]
    display(Markdown(part))


print(len(clips), "clips,", len(queries), "queries")

## 1. The corpus

43 short Pexels clips (4–8 s), each with the action it shows. 32 are *reversible*: played backwards they show
another real action (a person sits down ↔ stands up), so each has a forward and a reversed caption. 11 are
*one-way*: backwards they show something that cannot happen (a glass un-shatters).

The model sees a clip as frames sampled at 1 per second, each 640x360.

In [ ]:
def strip(frames, title):
    fig, axes = plt.subplots(1, len(frames), figsize=(2.2 * len(frames), 1.6))
    for ax, f in zip(axes, frames):
        ax.imshow(f)
        ax.axis("off")
    fig.suptitle(title, fontsize=10)
    plt.show()


clip = next(c for c in clips if c["key"] == "8435581")
frames = sample_frames(CLIPS / f"{clip['key']}_fwd.mp4")
strip(frames, f"{clip['caption_fwd']}  ({len(frames)} frames at 1 fps)")
print("reversed caption:", clip["caption_rev"])

## 2. One clip and one query, on the iGPU

`OVEmbedder` runs the INT8 model with OpenVINO: a vision graph turns each frame into 120 tokens, a 24-layer
encoder reads all of them together, and the mean of its outputs is the clip's 768-d unit vector. A query goes
through the same encoder, so the two vectors can be compared with a dot product.

In [ ]:
try:
    device = igpu()
except RuntimeError:
    device = "CPU"  # no Intel iGPU: everything below still runs, slower
model = OVEmbedder("int8", device)

t = time.perf_counter()
v = model.embed_videos([frames])[0]
t_clip = time.perf_counter() - t
t = time.perf_counter()
q = model.embed_text("a person sits down on a chair")
t_query = time.perf_counter() - t

print(f"{device}: clip -> vector {t_clip:.1f} s (first call includes compiling), query {t_query * 1000:.0f} ms")
print("vector shape", v.shape, "| cosine(clip, query) =", round(float(v @ q), 3))

The same search as printed in the post, in a few lines (it embeds all 43 clips again, about 100 s on the iGPU):

In [ ]:
from pathlib import Path

import numpy as np
from ov_model import OVEmbedder, igpu
from smoke_export import sample_frames

model = OVEmbedder("int8", igpu())  # INT8 on the Intel iGPU
paths = sorted(Path("data/clips").glob("*_fwd.mp4"))
V = np.stack([model.embed_videos([sample_frames(p)])[0] for p in paths])

q = model.embed_text("a laptop showing a world map")
scores = V @ q  # every vector has length 1, so this is the cosine
for i in np.argsort(-scores)[:3]:
    print(f"{scores[i]:.3f}  {paths[i].name}")

## 3. Search

Embed every clip once (saved in `index/`, as `search.py` does), then a search is one query embedding plus one
matrix-vector product.

In [ ]:
path = INDEX / f"search_int8_{'iGPU' if device != 'CPU' else 'CPU'}.npz"
if path.exists():
    V = np.load(path)["V"]
else:
    t = time.perf_counter()
    V = np.stack([model.embed_videos([sample_frames(CLIPS / f"{c['key']}_fwd.mp4")])[0] for c in clips])
    np.savez(path, V=V)
    print(f"indexed {len(V)} clips in {time.perf_counter() - t:.0f} s")


def search(text, k=3, dim=768):
    s = truncate(V, dim) @ truncate(model.embed_text(text), dim)
    top = np.argsort(-s)[:k]
    fig, axes = plt.subplots(1, k, figsize=(3 * k, 2))
    for ax, i in zip(axes, top):
        ax.imshow(sample_frames(CLIPS / f"{clips[i]['key']}_fwd.mp4")[0])
        ax.set_title(f"{s[i]:.3f}  {clips[i]['action']}", fontsize=8)
        ax.axis("off")
    fig.suptitle(f'"{text}"', fontsize=10)
    plt.show()


search("a man inflating a purple balloon")
search("a candle flame going out")

Over all 23 queries (scored by `evaluate.py` on the PyTorch reference, and by `benchmark.py` on every OpenVINO
configuration), the right clip comes first 22 times and is in the top 3 every time:

In [ ]:
section(OUT / "results_pytorch_fp32_fps1.md", "23 queries", "| Query")

## 4. Which way does time run?

Take the same frames and reverse their order. If the model reads order, the reversed clip should move towards
the reversed caption. The *paired* test asks exactly that, with everything the two versions share cancelled out:
does `v_fwd - v_rev` point the same way as `t_fwd - t_rev`?

In [ ]:
v_fwd = v
v_rev = model.embed_videos([frames[::-1].copy()])[0]
t_fwd = model.embed_text(clip["caption_fwd"])
t_rev = model.embed_text(clip["caption_rev"])

print(f"cosine(forward, reversed)  = {v_fwd @ v_rev:.4f}   (two different clips: ~0.61)")
print(f"forward caption -> forward {t_fwd @ v_fwd:.4f}, reversed {t_fwd @ v_rev:.4f}")
print(f"reversed caption -> forward {t_rev @ v_fwd:.4f}, reversed {t_rev @ v_rev:.4f}")
print("paired test right?", bool((v_fwd - v_rev) @ (t_fwd - t_rev) > 0))

One clip is one coin flip. Over the 32 reversible clips, and with a linear classifier trained to tell forward
from reversed vectors (one clip held out at a time), the result is chance:

In [ ]:
section(OUT / "results_pytorch_fp32_fps1.md", "### Reverse = same frames", "### Reverse = reversed file")

## 5. Where does the order go?

`order_probe.py` looks inside. Each frame's tokens *do* change with where the frame sits in the sequence (per-token
cosine 0.93 forward vs reversed, against 0.997 for the pooled vectors), but shuffling changes them about as much
as reversing, and no readout recovers the direction. Averaging every frame encoded on its own (a *bag of frames*)
searches as well as the whole clip.

In [ ]:
section(OUT / "order_probe.md", "## 1 fps", "## 2 fps")

## 6. A model built for video: VideoPrism

VideoPrism-LvT-B has attention layers across frames. It searches the same corpus 23/23, and it reacts when the
frames are shuffled, but reversed motion is still smooth motion: direction is at chance here too.

In [ ]:
section(OUT / "results_videoprism_lvt_b_16f.md", "### Reverse = same frames", "### Reverse = reversed file")
section(OUT / "results_videoprism_lvt_b_16f.md", "How far apart", "Filmed mirrors")

## 7. What it costs on the laptop

Three benchmark sessions on an i7-12700H with Iris Xe; bars are the median, whiskers the min–max across sessions.

In [ ]:
display(Image(filename=str(OUT / "latency.png")))
section(OUT / "benchmark.md", "| Model")

## 8. Do you need a vector database yet?

At this size, no: an exact search over 43 vectors takes microseconds. `scale.py` times it up to a million vectors
on one CPU thread, and checks that cutting the vectors to 256 dimensions (the model is trained for it) keeps the
search intact here.

In [ ]:
section(OUT / "scale.md", "| Vectors")

In [ ]:
search("a woman putting on glasses", dim=256)